## Amazon Product Recommendation System

In [14]:
import pandas as pd
import numpy as np
import re
import ast
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [15]:
df = pd.read_csv("amazon.csv")

print("Shape of dataset:", df.shape)
print(df.head())


Shape of dataset: (1465, 16)
   product_id                                       product_name  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightning Fast Cha...   
1  B098NS6PVG  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...   
2  B096MSW6CT  Sounce Fast Phone Charging Cable & Data Sync U...   
3  B08HDJ86NZ  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...   
4  B08CF3B7N1  Portronics Konnect L 1.2M Fast Charging 3A 8 P...   

                                            category discounted_price  \
0  Computers&Accessories|Accessories&Peripherals|...             ₹399   
1  Computers&Accessories|Accessories&Peripherals|...             ₹199   
2  Computers&Accessories|Accessories&Peripherals|...             ₹199   
3  Computers&Accessories|Accessories&Peripherals|...             ₹329   
4  Computers&Accessories|Accessories&Peripherals|...             ₹154   

  actual_price discount_percentage rating rating_count  \
0       ₹1,099                 64%    4.2       24,269   
1      

In [16]:
cols_to_use = ["product_id", "product_name", "category", "discounted_price", "actual_price", "discount_percentage", "rating",
               "rating_count", "about_product", "review_title", "review_content", "img_link", "product_link"]

df = df[cols_to_use].copy()

# Drop duplicates
df.drop_duplicates(subset=["product_id"], inplace=True)

# Drop rows where important text fields are missing
df.dropna(subset=["product_name", "category", "about_product"], inplace=True)

In [17]:
def clean_price(x):
    try:
        x = str(x).replace("₹", "").replace(",", "").strip()
        return float(x)
    except:
        return np.nan

def clean_percentage(x):
    try:
        x = str(x).replace("%", "").strip()
        return float(x)
    except:
        return np.nan

def clean_rating(x):
    try:
        x = str(x).strip()
        if x == "|" or x == "":
            return np.nan
        return float(x)
    except:
        return np.nan

def clean_rating_count(x):
    try:
        x = str(x).replace(",", "").strip()
        return float(x)
    except:
        return np.nan

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [18]:
df["discounted_price"] = df["discounted_price"].apply(clean_price)
df["actual_price"] = df["actual_price"].apply(clean_price)
df["discount_percentage"] = df["discount_percentage"].apply(clean_percentage)
df["rating"] = df["rating"].apply(clean_rating)
df["rating_count"] = df["rating_count"].apply(clean_rating_count)


df["discounted_price"].fillna(df["discounted_price"].median(), inplace=True)
df["actual_price"].fillna(df["actual_price"].median(), inplace=True)
df["discount_percentage"].fillna(df["discount_percentage"].median(), inplace=True)
df["rating"].fillna(df["rating"].median(), inplace=True)
df["rating_count"].fillna(df["rating_count"].median(), inplace=True)


df["product_name_clean"] = df["product_name"].apply(clean_text)
df["category_clean"] = df["category"].apply(lambda x: clean_text(str(x).replace("|", " ")))
df["about_product_clean"] = df["about_product"].apply(clean_text)
df["review_title_clean"] = df["review_title"].fillna("").apply(clean_text)
df["review_content_clean"] = df["review_content"].fillna("").apply(clean_text)

In [19]:
df["combined_text"] = (
    df["product_name_clean"] + " " +
    df["category_clean"] + " " +
    df["about_product_clean"] + " " +
    df["review_title_clean"] + " " +
    df["review_content_clean"]
)

print("Shape after cleaning:", df.shape)


# TF-IDF Vectorization

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

Shape after cleaning: (1351, 19)
TF-IDF matrix shape: (1351, 5000)


In [20]:
df["combined_text"].head(2)

0    wayona nylon braided usb to lightning fast cha...
1    ambrane unbreakable 60w 3a fast charging 1 5m ...
Name: combined_text, dtype: object

In [21]:
# Build Index Mapping
# product_name to index

indices = pd.Series(df.index, index=df["product_name"]).drop_duplicates()


# Normalize rating and rating_count for reranking

scaler = MinMaxScaler()

df[["rating_norm", "rating_count_norm"]] = scaler.fit_transform(
    df[["rating", "rating_count"]]
)

# Hybrid ranking score weights
TEXT_WEIGHT = 0.75
RATING_WEIGHT = 0.15
POPULARITY_WEIGHT = 0.10

In [22]:
# Recommendation Function

def recommend_products(product_name, top_n=5):
    if product_name not in indices:
        return f"Product '{product_name}' not found in dataset."

    idx = indices[product_name]

    # Pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Exclude the same product
    sim_scores = sim_scores[1: top_n + 20]

    recommendations = []
    seen_ids = set()

    for i, sim_score in sim_scores:
        product_id = df.iloc[i]["product_id"]
        if product_id in seen_ids:
            continue

        seen_ids.add(product_id)

        hybrid_score = (
            TEXT_WEIGHT * sim_score +
            RATING_WEIGHT * df.iloc[i]["rating_norm"] +
            POPULARITY_WEIGHT * df.iloc[i]["rating_count_norm"]
        )

        recommendations.append({
            "product_name": df.iloc[i]["product_name"],
            "category": df.iloc[i]["category"],
            "rating": df.iloc[i]["rating"],
            "rating_count": int(df.iloc[i]["rating_count"]),
            "discounted_price": df.iloc[i]["discounted_price"],
            "similarity_score": round(sim_score, 4),
            "hybrid_score": round(hybrid_score, 4),
            "product_link": df.iloc[i]["product_link"]
        })

    rec_df = pd.DataFrame(recommendations)

    # Sort by hybrid score
    rec_df = rec_df.sort_values(by="hybrid_score", ascending=False).head(top_n).reset_index(drop=True)

    return rec_df

In [23]:
#  Search Function

def search_products(keyword, top_n=10):
    keyword = clean_text(keyword)

    mask = (
        df["product_name_clean"].str.contains(keyword, na=False) |
        df["category_clean"].str.contains(keyword, na=False) |
        df["about_product_clean"].str.contains(keyword, na=False)
    )

    results = df.loc[mask, ["product_name", "category", "rating", "discounted_price"]].copy()
    results = results.sort_values(by="rating", ascending=False).head(top_n).reset_index(drop=True)

    return results


# Example Usage

print("\nSample search results:\n")
print(search_products("earphone", top_n=5))

# Replace with any exact product name from your dataset
sample_product = df["product_name"].iloc[0]

print("\nSelected product:\n", sample_product)

print("\nRecommended products:\n")
print(recommend_products(sample_product, top_n=5))


# Optional Interactive Mode

while True:
    user_input = input("\nEnter a product name to get recommendations (or type 'exit'): ")

    if user_input.lower() == "exit":
        print("Exiting recommender system.")
        break

    result = recommend_products(user_input, top_n=5)

    print("\nRecommendations:\n")
    print(result)


Sample search results:

                                        product_name  \
0  AirCase Rugged Hard Drive Case for 2.5-inch We...   
1  Gizga Essentials Hard Drive Case Shell, 6.35cm...   
2  Gizga Essentials Earphone Carrying Case, Multi...   
3  Redmi 80 cm (32 inches) Android 11 Series HD R...   
4  OPPO A31 (Mystery Black, 6GB RAM, 128GB Storag...   

                                            category  rating  discounted_price  
0  Computers&Accessories|Accessories&Peripherals|...     4.5             299.0  
1  Computers&Accessories|Accessories&Peripherals|...     4.5             199.0  
2   Electronics|Headphones,Earbuds&Accessories|Cases     4.3             119.0  
3  Electronics|HomeTheater,TV&Video|Televisions|S...     4.2           13999.0  
4  Electronics|Mobiles&Accessories|Smartphones&Ba...     4.2           12490.0  

Selected product:
 Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pr

Exiting recommender system.


In [24]:
import pickle

recommender_data = {
    "df": df,
    "cosine_sim": cosine_sim,
    "indices": indices
}

with open("recommender_model.pkl", "wb") as f:
    pickle.dump(recommender_data, f)

print("Model saved successfully!")

Model saved successfully!
